In [1]:
import jax
import jax.numpy as jnp

from matfree import stochtrace, decomp

In [12]:
_key = lambda x: jax.random.PRNGKey(x)

D = 100
rank = 32
num_matvecs = 75
num_samples = 10

def rank1(key, d):
    eps = jax.random.normal(key, (d,1))
    return eps@eps.T

def sum_of_rank1s(key, rank, d):
    keys = jax.random.split(key, rank)
    return jax.vmap(lambda __key: rank1(__key, d))(keys).sum(axis=0)

A = jax.random.normal(_key(442), (D,D))
X = A @ A.T

# X = jnp.eye(D) + 1 * sum_of_rank1s(_key(442), rank, D)

tridiag = decomp.tridiag_sym(num_matvecs)

v = jax.random.normal(_key(2232), (D))

def matvec(v):
    return X@v

res = tridiag(matvec, v)
Q = res.Q_tall
T = res.J_small

head = jnp.trace(T)

# Residual Hutchinson
def proj(v): return v - Q @ (Q.T @ v)
def res_matvec(v): return proj(X @ proj(v))

problem = stochtrace.integrand_trace()
sampler = stochtrace.sampler_normal(v, num=num_samples)
estimate = stochtrace.estimator(problem, sampler)
tail = estimate(res_matvec, key=_key(123))

est = head + tail
print("Hutch# trace estimate:", est)
print("True trace:", jnp.trace(X))

Hutch# trace estimate: 10012.023
True trace: 9964.119


# Tests with a real GGN

In [13]:
# load MAP state

import optax

from src.utils import load_checkpoint, load_yaml, count_model_params
from src.scalemodels import TrainState, EMPTY_STATS
from src.toymodels import SimpleClassifier
from src.toydata import get_dataloaders

model_name = 'toyclassifier_banana'
cfg_path = f'config/toy/{model_name}.yml'
dataset = 'banana'

cfg = load_yaml(cfg_path)
model_cfg = cfg['model']
opt_cfg = cfg['optimization']
alpha = opt_cfg["alpha"]
map_cfg = opt_cfg["map"]

model_type = model_cfg.get("name", "regressor")  # 'regressor' or 'classifier'
num_h = model_cfg["num_h"]
num_l = model_cfg["num_l"]
num_c = model_cfg.get("num_c", 2) if model_type == "classifier" else 1
rng_model = jax.random.PRNGKey(model_cfg["seed"])
map_batch_size = map_cfg["batch_size"]
epochs_map = map_cfg["epochs"]
lr_map = map_cfg["lr"]

model = SimpleClassifier(numh=num_h, numl=num_l, numc=num_c)

train_loader, test_loader, _ = get_dataloaders(dataset=dataset, batch_size=map_batch_size)

dummy_input = next(iter(train_loader))[0][:1]
variables = model.init(rng_model, dummy_input)
optimizer_map = optax.adam(1e-3)
model_state = TrainState.create(
    apply_fn=model.apply,
    params=variables['params'],
    tx=optimizer_map,
    batch_stats = variables.get('batch_stats', EMPTY_STATS),
)
map_ckpt_prefix = f"map_{dataset}"

map_state = load_checkpoint(
    ckpt_dir="checkpoint/map/",
    prefix=map_ckpt_prefix,
    target=model_state
)

D = count_model_params(variables)

[checkpoint] Loaded model checkpoint from /Users/nielsraunkjaer/Desktop/thesis/laplace-inducing-points/checkpoint/map (prefix=map_banana)


In [17]:
# compute W and GGN

from src.ggn import compute_W_vps, compute_ggn_vp, compute_ggn_dense
from src.utils import flatten_nn_params

flat_params, unravel_fn = flatten_nn_params(map_state.params)

FULL_DATA  = next(iter(train_loader))[0] # 32 samples
# GGN_DATA   = FULL_DATA[:-1]
# EXTRA_TERM = FULL_DATA[None,-1]

GGN_full_dense, *_ =  compute_ggn_dense(map_state, FULL_DATA, model_type=model_type, flat_params=flat_params, unravel_fn=unravel_fn, full_set_size=None)
# GGN_dense, *_      =  compute_ggn_dense(map_state, GGN_DATA,  model_type=model_type, flat_params=flat_params, unravel_fn=unravel_fn, full_set_size=None)

GGN_full =  compute_ggn_vp(map_state, FULL_DATA,  model_type=model_type, flat_params=flat_params, unravel_fn=unravel_fn, full_set_size=None)
# GGN      =  compute_ggn_vp(map_state, GGN_DATA,   model_type=model_type, flat_params=flat_params, unravel_fn=unravel_fn, full_set_size=None)
# W, WT    =  compute_W_vps( map_state, GGN_DATA,   model_type=model_type, flat_params=flat_params, unravel_fn=unravel_fn, full_set_size=None)
# Wp, WTp  =  compute_W_vps( map_state, EXTRA_TERM, model_type=model_type, flat_params=flat_params, unravel_fn=unravel_fn, full_set_size=None)

In [18]:
v = jax.random.normal(_key(53113), (D,))

tridiag = decomp.tridiag_sym(16)

res = tridiag(GGN_full, v)
Q = res.Q_tall
T = res.J_small

#*========*
#* Hutch# *
#*========*
head = jnp.trace(T)

def proj(v): 
    return v - Q @ (Q.T @ v)

def res_matvec(v): 
    t = GGN_full(proj(v))
    return proj(t)

problem = stochtrace.integrand_trace()
sampler = stochtrace.sampler_normal(v, num=1)
estimate = stochtrace.estimator(problem, sampler)
tail = estimate(res_matvec, key=_key(123))

est = head + tail
print("Hutch# trace estimate:", est)
print("True trace:", jnp.trace(GGN_full_dense))

Hutch# trace estimate: 617.62836
True trace: 617.982
